# LSTM Check — проверка размерностей

In [1]:
import sys
sys.path.append('..')
import torch
import pandas as pd
from torch.utils.data import DataLoader
from src.models.dataset import FitnessSequenceDataset
from src.models.lstm import LSTMRegressor
print('Imports OK')

Imports OK


## Загрузка данных и создание Dataset

In [2]:
df = pd.read_parquet('../data/processed/train_features.parquet')
n = len(df)
split_idx = int(n * 0.8)
train_df = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()
train_dataset = FitnessSequenceDataset(train_df, sequence_length=7, fit_scaler=True)
val_dataset = FitnessSequenceDataset(val_df, sequence_length=7, scaler=train_dataset.scaler)
print(f'Train sequences: {len(train_dataset)}')
print(f'Val sequences: {len(val_dataset)}')

Scaler сохранён: models\scaler.pkl
Dataset: 77 sequences, shape=(77, 7, 17)
Dataset: 15 sequences, shape=(15, 7, 17)
Train sequences: 77
Val sequences: 15


## Проверка одного элемента

In [3]:
X_sample, y_sample = train_dataset[0]
print(f'X shape: {X_sample.shape}')
print(f'y shape: {y_sample.shape}')
print(f'NaN в X: {torch.isnan(X_sample).any()}')
print(f'NaN в y: {torch.isnan(y_sample).any()}')

X shape: torch.Size([7, 17])
y shape: torch.Size([1])
NaN в X: False
NaN в y: False


## Проверка DataLoader батча

In [4]:
loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
X_batch, y_batch = next(iter(loader))
print(f'Batch X shape: {X_batch.shape}')
print(f'Batch y shape: {y_batch.shape}')

Batch X shape: torch.Size([32, 7, 17])
Batch y shape: torch.Size([32, 1])


## Проверка модели

In [5]:
n_features = X_batch.shape[2]
model = LSTMRegressor(input_size=n_features, hidden_size=64, num_layers=2, dropout=0.2)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Всего параметров: {total_params:,}')

LSTMRegressor(
  (lstm): LSTM(17, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)
Всего параметров: 54,593


In [6]:
model.eval()
with torch.no_grad():
    preds = model(X_batch)
print(f'Input:  {X_batch.shape}')
print(f'Output: {preds.shape}')
print(f'NaN в предсказаниях: {torch.isnan(preds).any()}')
print(f'Пример предсказаний: {preds[:5].squeeze().tolist()}')

Input:  torch.Size([32, 7, 17])
Output: torch.Size([32, 1])
NaN в предсказаниях: False
Пример предсказаний: [0.03989846631884575, 0.054730940610170364, 0.025116145610809326, 0.05863553285598755, 0.02385692484676838]
